# Gradient boosting in practice and interpretation

Three open-source libraries implement fast, regularised gradient boosting and dominate tabular machine learning: **XGBoost**, **LightGBM** and **CatBoost**. This notebook covers the second block of Session 10. Sections 1–5 show what distinguishes the three libraries, how they handle categorical features, how class weights and early stopping work, and how to tune them without overfitting the validation data. The lecture of block 1 ([01-lecture-from-one-tree-to-an-ensemble.ipynb](01-lecture-from-one-tree-to-an-ensemble.ipynb)) explained the underlying algorithm. Sections 6–9 ask what a boosted model has learned and whether it is worth using: **feature importance** (impurity-based), **permutation importance**, **SHAP values** as a tool for checking and debugging models, and the question every model update raises: is the new **challenger** model really better than the **current** model, by enough to replace it? Sections 1–8 use the Telco churn data; Section 9 applies everything to a practical tabular question: what drives the nightly price of a Berlin Airbnb listing, and does a tree ensemble price better than the linear model of Session 6?

> **Note.** The code cells in this notebook build on each other; run them in order. They need the Telco data from GitHub, the Inside Airbnb data for Section 9 (`uv run python case-study/prepare_airbnb.py`) and the packages `xgboost`, `lightgbm`, `catboost` and `shap`, which are part of the course environment. **On macOS**, XGBoost and LightGBM need the OpenMP runtime: run `brew install libomp` once, otherwise `import xgboost` fails with a message about `libomp.dylib`.

```mermaid
flowchart TD
    s["Gradient boosting<br/>on a table"] --> q1{"Many categorical<br/>columns?"}
    q1 -->|yes| cb["CatBoost<br/>cat_features=..."]
    q1 -->|no| q2{"Large data,<br/>speed matters?"}
    q2 -->|yes| lgb["LightGBM or<br/>HistGradientBoosting"]
    q2 -->|no| any["Any of the three;<br/>tune, then compare"]
    cb --> es["Early stopping on<br/>a validation split"]
    lgb --> es
    any --> es
    es --> cv["Compare with CV<br/>against the baseline"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## 1. XGBoost, LightGBM and CatBoost

### Concept

All three libraries fit the gradient boosting model of the block 1 lecture, with additions:

| | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| Origin | Chen & Guestrin (2016), University of Washington | Ke et al. (2017), Microsoft | Prokhorenkova et al. (2018), Yandex |
| Tree growth | level-wise (depth by depth) by default | **leaf-wise**: always splits the leaf with the largest gain | **symmetric** (oblivious) trees: the same split on a whole level |
| Main complexity parameter | `max_depth` | `num_leaves` | `depth` |
| Split search | histogram (default) or exact | histogram | histogram |
| Categories | `enable_categorical=True` with pandas `category` dtype | pandas `category` dtype | `cat_features=[...]`, ordered target statistics |
| scikit-learn class | `XGBClassifier` | `LGBMClassifier` | `CatBoostClassifier` |

Common additions to plain gradient boosting:

- **Regularisation** of the leaf values (L1 and L2 penalties, `reg_alpha`, `reg_lambda`) and a minimum gain or minimum number of rows per leaf (`min_child_weight`, `min_child_samples`).
- **Row and column subsampling** per tree (`subsample`, `colsample_bytree`), borrowed from random forests: each tree sees a random part of the data, which reduces variance (*stochastic gradient boosting*).
- **Missing values** are handled natively: each split learns in which direction missing values go.

All three have scikit-learn interfaces (`fit`, `predict`, `predict_proba`), so they work in `Pipeline`, `cross_val_score` and `GridSearchCV`.

### Why it matters

These libraries are the standard tools for tabular prediction in industry and in competitions. Knowing their main parameters and their few differences is enough for most projects; which of the three wins on a given dataset is an empirical question, answered by cross-validation.

### How it works in Python

The libraries can use the categorical columns directly, without one-hot encoding:

In [2]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
df = pd.read_csv(URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
cat = X.select_dtypes(exclude="number").columns.tolist()          # 16 categorical columns
X_cat = X.astype({c: "category" for c in cat})                     # pandas category dtype
Xc_tr, Xc_te, y_tr, y_te = train_test_split(X_cat, y, test_size=0.25, stratify=y, random_state=0)

models = {
    "xgboost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, enable_categorical=True),
    "lightgbm": LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=8, verbose=-1),
    "catboost": CatBoostClassifier(n_estimators=300, learning_rate=0.05, depth=4, cat_features=cat,
                                   verbose=0, allow_writing_files=False),
}
for name, model in models.items():
    # CatBoost takes the original string columns; the other two take category dtype
    data_tr, data_te = (X.loc[Xc_tr.index], X.loc[Xc_te.index]) if name == "catboost" else (Xc_tr, Xc_te)
    model.fit(data_tr, y_tr)
    print(name, round(roc_auc_score(y_te, model.predict_proba(data_te)[:, 1]), 3))
# xgboost 0.843
# lightgbm 0.842
# catboost 0.848

xgboost 0.843


lightgbm 0.842


catboost 0.848


The split is the same as in the block 1 lecture (same `random_state`, same rows), so the scores are comparable: all three libraries land between 0.842 and 0.848, next to the random forest (0.844) and the logistic regression (0.844). On a small dataset with a few strong effects, the choice of library matters much less than the features.

### In practice

- Chen and Guestrin (2016) report that XGBoost was used in 17 of the 29 winning solutions published on Kaggle's blog in 2015.
- CatBoost was developed at Yandex, where gradient-boosted trees had long been used for search ranking; its paper (Prokhorenkova et al., 2018) motivates ordered boosting with the target leakage of naive category encodings.
- LightGBM's speed on large data made it the most common model in the M5 forecasting competition on Walmart sales (Makridakis et al., 2022).

> **Warning.** Comparing libraries at their default settings says little: defaults differ in the number of trees, depth and learning rate. Compare after tuning each, or with equal settings as above, and report the variation across folds.

## 2. Categorical features

### Concept

scikit-learn's `DecisionTreeClassifier` needs numbers, so categories are one-hot encoded: each dummy column allows only splits of the form "category A versus all others". The boosting libraries can split a categorical feature **natively** into two *groups* of categories:

- **LightGBM and XGBoost** sort the categories of a node by their gradient statistics (roughly, by their current mean residual) and search the best cut in that order, as for a numeric feature. This finds splits such as {Month-to-month} versus {One year, Two year} in one step.
- **CatBoost** replaces each category by an **ordered target statistic**: the rows are put in a random order, and each row's category is encoded with the (smoothed) mean target of the *earlier* rows with the same category. A row never sees its own label, the same principle as the cross-fitted target encoding of Session 9.

For a column with 3 levels, the difference is small. For a column with hundreds or thousands of levels (store, product, postcode), one-hot encoding creates very wide, sparse data; native handling or target encoding usually works better.

### Why it matters

Real tables are full of categorical columns. Handling them natively saves preprocessing code, and the training–serving pipeline stays shorter. But each library expects categories in a specific form; getting it wrong silently changes the model.

### How it works in Python

In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(5, shuffle=True, random_state=0)
X_oh = pd.get_dummies(X, columns=cat, drop_first=True, dtype=int)
params = dict(n_estimators=300, learning_rate=0.05, num_leaves=8, verbose=-1, n_jobs=1)
for name, data in [("one-hot", X_oh), ("native categories", X_cat)]:
    scores = cross_val_score(LGBMClassifier(**params), data, y, cv=cv, scoring="roc_auc")
    print(name, data.shape[1], round(scores.mean(), 3))
# one-hot 30 0.845
# native categories 19 0.845

one-hot 30 0.845


native categories 19 0.845


On the Telco data, both encodings give the same score; the categorical columns have at most four levels. The workbook [extras/10-gradient-boosting-categorical.ipynb](extras/10-gradient-boosting-categorical.ipynb) (scikit-learn gallery) compares one-hot, ordinal and native encodings on a dataset with more levels.

> **Caution.** Pass categories with the **same categories in training and test data**. `astype("category")` on the test set alone can number the levels differently. Convert once on the whole feature table before splitting (the list of levels is not a leak), or use `pd.Categorical(values, categories=train_levels)`. String columns passed to XGBoost or LightGBM raise an error or are rejected; CatBoost needs `cat_features`.

> 📖 **Read more:** [extras/12-catboost-tutorial.ipynb](extras/12-catboost-tutorial.ipynb) (CatBoost tutorials): CatBoost's own tutorial on the Titanic data: `cat_features`, built-in cross-validation, early stopping with the best iteration, feature importances and parameter tuning with `hyperopt` (needs `hyperopt`).

## 3. Class weights

### Concept

As in Session 8, a **class weight** multiplies the loss of each row of a class. The libraries use different parameter names:

| Library | Parameter | Typical value |
|---|---|---|
| scikit-learn, LightGBM | `class_weight="balanced"` | n / (K · n_c) |
| XGBoost (binary) | `scale_pos_weight` | (number of negatives) / (number of positives) |
| CatBoost | `auto_class_weights="Balanced"` | n / (K · n_c) |

For the churn training data with 3,880 stayers and 1,402 churners, `scale_pos_weight` = 3,880 / 1,402 = 2.77.

### Why it matters

Weights move the predicted probabilities of the minority class upwards. The model then predicts churn more often: recall rises, precision falls. The **ranking** of customers changes hardly at all, so ROC AUC stays nearly the same. Weights are therefore a way of moving the decision rule, comparable to lowering the threshold (Session 8).

### How it works in Python

In [4]:
from sklearn.metrics import precision_score, recall_score

ratio = (y_tr == 0).sum() / (y_tr == 1).sum()
print(round(ratio, 2))                                            # 2.77
for w in [1, ratio]:
    m = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, enable_categorical=True,
                      scale_pos_weight=w).fit(Xc_tr, y_tr)
    p = m.predict(Xc_te)
    print(round(w, 2), round(precision_score(y_te, p), 3), round(recall_score(y_te, p), 3),
          round(roc_auc_score(y_te, m.predict_proba(Xc_te)[:, 1]), 3))
# weight precision recall AUC
# 1      0.67      0.518  0.845
# 2.77   0.518     0.79   0.845

2.77


1 0.67 0.518 0.845


2.77 0.518 0.79 0.845


Without weights the model finds 52 % of the churners, and 67 % of its alarms are correct. With weight 2.77 it finds 79 % of the churners, but only 52 % of its alarms are correct. ROC AUC is identical: the weights changed the operating point, not the ranking.

> **Warning.** Weighted models give shifted probabilities. If the business needs calibrated churn probabilities (for an expected-loss calculation), fit without weights and choose the threshold from the costs, or recalibrate.

## 4. Early stopping

### Concept

The number of trees is the most important boosting hyperparameter: too few underfit, too many overfit. **Early stopping** decides it automatically. A **validation set** is split off the *training* data. After each boosting round, the library computes the loss on the validation set; when it has not improved for a given number of rounds (`early_stopping_rounds`), training stops and the model keeps the trees up to the best round (`best_iteration`).

### Why it matters

Early stopping saves computation and removes one hyperparameter from the search. It is the usual way to set the number of trees, especially with a high learning rate, and the standard way to stop training neural networks as well.

### How it works in Python

In [5]:
import lightgbm

# hold out 20 % of the TRAINING data; the test set stays untouched
Xf_tr, Xf_va, yf_tr, yf_va = train_test_split(Xc_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=1)

xgb = XGBClassifier(n_estimators=300, learning_rate=0.3, max_depth=4, enable_categorical=True,
                    eval_metric="logloss", early_stopping_rounds=20)
xgb.fit(Xf_tr, yf_tr, eval_set=[(Xf_va, yf_va)], verbose=False)
curve = xgb.evals_result()["validation_0"]["logloss"]
print(xgb.best_iteration, round(min(curve), 3), len(curve))   # 25 0.383 46: best round, its loss, rounds run
print(round(roc_auc_score(y_te, xgb.predict_proba(Xc_te)[:, 1]), 3))   # 0.841

lgbm = LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=8, verbose=-1)
lgbm.fit(Xf_tr, yf_tr, eval_X=Xf_va, eval_y=yf_va,       # LightGBM < 4.7: eval_set=[(Xf_va, yf_va)]
         callbacks=[lightgbm.early_stopping(50, verbose=False)])
print(lgbm.best_iteration_, round(roc_auc_score(y_te, lgbm.predict_proba(Xc_te)[:, 1]), 3))   # 207 0.843

25 0.383 46
0.841


207 0.843


With the high learning rate 0.3, XGBoost's validation log loss reaches its minimum of 0.383 at round 25 (26 trees, counting from 0) and rises afterwards; training stopped 20 rounds later. LightGBM with the smaller learning rate 0.05 needs 208 trees. Both reach a test AUC similar to the fixed 300-tree models. CatBoost accepts `eval_set` and `early_stopping_rounds` in `fit`; `HistGradientBoostingClassifier` has `early_stopping=True` with an internal `validation_fraction`. The workbook [extras/09-gradient-boosting-early-stopping.ipynb](extras/09-gradient-boosting-early-stopping.ipynb) plots the curves.

> **Caution.** Never use the test set as `eval_set`. The number of trees is then chosen on the test data, and the test score is no longer an independent estimate. For time-ordered data (the monthly reviews of Session 12), the validation set must be *later* than the training rows.

## 5. Tuning

### Concept

The parameters that matter most, roughly in this order:

1. **Number of trees and learning rate** together: fix a small learning rate (0.03–0.1) and set the number of trees by early stopping or search.
2. **Tree complexity**: `max_depth` (XGBoost, CatBoost `depth`) or `num_leaves` (LightGBM), and the minimum rows per leaf (`min_child_samples`, `min_child_weight`).
3. **Subsampling**: `subsample` (rows per tree) and `colsample_bytree` (features per tree), typically 0.5–1.0.
4. **Regularisation**: `reg_lambda`, `reg_alpha`.

With many parameters, **randomised search** (`RandomizedSearchCV`, Session 7) is more efficient than a grid: it samples combinations from distributions, for example a log-uniform learning rate. The best cross-validated score of the search is optimistic, because it is the maximum over many tries; report the score on held-out data (or nested CV) next to it.

### Why it matters

Untuned boosting with too many leaves or too high a learning rate can be worse than a random forest. A modest search usually gains a little; on the churn data, very little. Knowing *when* tuning is not worth it is also a result.

### How it works in Python

In [6]:
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    LGBMClassifier(n_estimators=200, subsample_freq=1, n_jobs=1, verbose=-1, random_state=0),
    {"learning_rate": loguniform(0.01, 0.3),     # sampled on a log scale
     "num_leaves": randint(4, 64),
     "min_child_samples": randint(5, 100),
     "subsample": uniform(0.5, 0.5)},           # uniform between 0.5 and 1.0
    n_iter=20, scoring="roc_auc", cv=StratifiedKFold(5, shuffle=True, random_state=0), random_state=0,
).fit(Xc_tr, y_tr)
print({k: round(float(v), 3) for k, v in search.best_params_.items()})
# {'learning_rate': 0.034, 'min_child_samples': 6.0, 'num_leaves': 5.0, 'subsample': 0.55}
print(round(search.best_score_, 3), round(roc_auc_score(y_te, search.predict_proba(Xc_te)[:, 1]), 3))
# 0.85 0.847: best CV score and held-out test score

{'learning_rate': 0.034, 'min_child_samples': 6.0, 'num_leaves': 5.0, 'subsample': 0.55}
0.85 0.847


The search prefers very small trees (5 leaves) and a low learning rate, which fits the observation that the churn effects are simple. The held-out score (0.847) is close to the cross-validated one (0.850); the gain over the untuned LightGBM (0.842) is small.

> **Tip.** Set `n_jobs=1` in the LightGBM or XGBoost estimator when the search itself runs in parallel (`RandomizedSearchCV(..., n_jobs=-1)`). Both libraries use all CPU cores by default, and two levels of parallelism slow each other down or, on some systems, hang.

### In practice

- The LightGBM parameter-tuning guide recommends a small `learning_rate` with many iterations for accuracy, and names `num_leaves`, `min_data_in_leaf` and `max_depth` as the main parameters against overfitting.
- Optuna (Akiba et al., 2019), an open-source hyperparameter optimisation framework from Preferred Networks, provides a step-wise tuner for LightGBM that searches these parameters one group after another.

> **Warning.** Early stopping inside `RandomizedSearchCV` is tricky: the validation set for early stopping would have to be split off each training fold. Either fix the number of trees in the search (as above) or search with a fixed learning rate and use early stopping only for the final fit.

> **Caution.** **Many classes, few rows per class.** A multiclass booster fits one tree per class and round. For a class with few rows, the second derivative of the loss (the hessian) in a leaf is close to zero, and the Newton step of the leaf value, gradient divided by hessian, explodes. Raise the minimum hessian per leaf (`min_sum_hessian_in_leaf` in LightGBM, default 0.001, for example to 1.0) or add L2 regularisation of the leaf values (`l2_regularization` in `HistGradientBoostingClassifier`, default 0, for example 1.0). If a multiclass booster is worse than a constant prediction, suspect this first.

> 📖 **Read more:** [extras/11-ensemble-hyperparameters.ipynb](extras/11-ensemble-hyperparameters.ipynb) (INRIA MOOC): which settings matter for random forests (`max_features`, `max_leaf_nodes`, `min_samples_leaf`; more trees need no tuning) and why the settings of histogram gradient boosting (`max_iter`, `learning_rate`, `max_leaf_nodes`) must be tuned together, with `RandomizedSearchCV` on the California housing data.

A boosted model with 300 trees cannot be read like the depth-2 tree of block 1. Yet we must know what it has learned: to check that it uses sensible information, to find leaks and bugs, and to explain it to the people who rely on it. The code of Sections 6–9 starts again from the data, so it can also be run on its own after the first cell of this notebook.

```mermaid
flowchart LR
    m["Fitted model"] --> g{"Question"}
    g -->|"Which features does<br/>the model rely on?"| p["Permutation importance<br/>on held-out data"]
    g -->|"Why this prediction<br/>for this customer?"| s["SHAP values<br/>(local)"]
    g -->|"Is something wrong?"| d["SHAP summary:<br/>leak, bug, proxy?"]
    p --> c["Check against<br/>domain knowledge"]
    s --> c
    d --> c
```

## 6. Feature importance: impurity-based and permutation importance

### Concept

**Impurity-based importance** (`feature_importances_` in scikit-learn, "gain" in the boosting libraries) adds up, for each feature, the decrease in impurity of all splits that use it, weighted by the number of rows reaching the split. It is computed from the training data during fitting and comes for free. It has two known biases: it favours features with many distinct values (continuous features, identifiers), because they offer many possible thresholds, and it measures how much the feature was *used on the training data*, including to fit noise.

**Permutation importance** (Breiman, 2001) measures how much a model's score drops when the values of one feature are randomly shuffled, which breaks the link between that feature and the target while keeping its distribution:

1. Compute the score on a dataset (preferably held-out data).
2. For feature j: shuffle column j, recompute the score; the importance is the drop. Repeat several times and average.

Worked example: a model has a test ROC AUC of 0.842. Shuffling `Contract` lowers it to 0.754; the permutation importance of `Contract` is 0.842 − 0.754 = 0.088. Shuffling `gender` leaves it at 0.842: importance 0; the model does not rely on it.

### Why it matters

Importance answers "which inputs does the model rely on?" Computed on held-out data, permutation importance shows what *generalises*; computed on training data, it shows what the model *memorised*. Comparing the two is a quick overfitting check.

### How it works in Python

To expose the bias, we add a column of random numbers, `random_id`, which carries no information about churn:

In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
df = pd.read_csv(URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
cat = X.select_dtypes(exclude="number").columns.tolist()
X_cat = X.astype({c: "category" for c in cat})
Xc_tr, Xc_te, y_tr, y_te = train_test_split(X_cat, y, test_size=0.25, stratify=y, random_state=0)

X_oh = pd.get_dummies(X, columns=cat, drop_first=True, dtype=int)
rng = np.random.default_rng(0)
Xo_tr = X_oh.loc[Xc_tr.index].assign(random_id=rng.random(len(Xc_tr)))   # pure noise column
Xo_te = X_oh.loc[Xc_te.index].assign(random_id=rng.random(len(Xc_te)))

rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xo_tr, y_tr)  # deep trees
imp = pd.DataFrame({"impurity": rf.feature_importances_}, index=Xo_tr.columns)
for name, (Xs, ys) in {"perm_train": (Xo_tr, y_tr), "perm_test": (Xo_te, y_te)}.items():
    imp[name] = permutation_importance(rf, Xs, ys, scoring="roc_auc", n_repeats=10,
                                       random_state=0, n_jobs=-1).importances_mean
print(imp.sort_values("impurity", ascending=False).head(6).round(3).to_string())
#                                 impurity  perm_train  perm_test
# TotalCharges                       0.164       0.015      0.012
# tenure                             0.152       0.024      0.027
# MonthlyCharges                     0.138       0.009      0.004
# random_id                          0.123       0.007     -0.000
# InternetService_Fiber optic        0.038       0.006      0.019
# PaymentMethod_Electronic check     0.036       0.009      0.003

                                impurity  perm_train  perm_test
TotalCharges                       0.164       0.015      0.012
tenure                             0.152       0.024      0.027
MonthlyCharges                     0.138       0.009      0.004
random_id                          0.123       0.007     -0.000
InternetService_Fiber optic        0.038       0.006      0.019
PaymentMethod_Electronic check     0.036       0.009      0.003


By impurity, the random column is the fourth most important feature, ahead of the internet service type: deep trees split on it again and again to separate individual training rows. Permutation importance on the test data correctly gives it 0.000, and it ranks fibre-optic internet (0.019) above the charges. The three continuous features dominate the impurity ranking because they offer the most split points.

For the boosted model, permutation importance works the same way and can use the original categorical columns:

In [8]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=8, verbose=-1, random_state=0)
lgbm.fit(Xc_tr, y_tr)
print(round(roc_auc_score(y_te, lgbm.predict_proba(Xc_te)[:, 1]), 3))          # 0.842
perm = permutation_importance(lgbm, Xc_te, y_te, scoring="roc_auc", n_repeats=10, random_state=0)
print(pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(4).round(3).to_dict())
# {'Contract': 0.088, 'tenure': 0.056, 'InternetService': 0.013, 'MonthlyCharges': 0.011}

0.842


{'Contract': 0.088, 'tenure': 0.056, 'InternetService': 0.013, 'MonthlyCharges': 0.011}


### In practice

- Strobl et al. (2007) demonstrated with simulated and biological data that impurity importance in random forests favours variables with many categories or values; the scikit-learn example in [extras/13-permutation-importance.ipynb](extras/13-permutation-importance.ipynb) reproduces it with a random column on the Titanic data.
- Model risk management guidance for banks, such as the US Federal Reserve's SR 11-7, requires that the drivers of a model are understood and documented; permutation importance is one of the standard tools for this.

> **Warning.** **Correlated features share importance.** `tenure` and `TotalCharges` (≈ tenure × monthly charges) carry similar information. Shuffling one barely hurts the model, because it can use the other, so both can look unimportant although together they matter. Check correlations, or permute correlated groups together.

> **Caution.** Importance describes the *model*, not the world. "Fibre-optic internet is important for the churn prediction" does not mean that fibre optic *causes* churn; it may be a proxy for price, region or customer type.

> 📖 **Read more:** [extras/14-feature-importance.ipynb](extras/14-feature-importance.ipynb) (INRIA MOOC): compares the coefficients of a linear model (and how much they vary), the impurity importance of a random forest and permutation importance on the California housing data, with a random column added as a check.

## 7. SHAP values as a model-checking tool

### Concept

**SHAP values** (SHapley Additive exPlanations; Lundberg & Lee, 2017) split *one* prediction into additive contributions of the features:

model output for customer i = base value + φ₁ + φ₂ + … + φ_p.

The **base value** is the average model output over the training data; φ_j is the contribution of feature j for this customer. For tree models the output is usually in **log-odds**, so contributions add up on the log-odds scale. The φ_j are **Shapley values** from cooperative game theory: the average change in the prediction when feature j is added, taken over all orders in which the features could be added. `shap.TreeExplainer` computes them exactly and quickly for tree ensembles (Lundberg et al., 2020).

Worked example (log-odds), the first test customer of the code below: the base value is −1.718, the average log-odds over the training data (it corresponds to 15 %, less than the churn rate of 26.5 %, because the average of log-odds is not the log-odds of the average). For this customer `Contract = Month-to-month` adds +0.719, `InternetService = DSL` adds −0.342, `tenure = 11` adds +0.317, `PaymentMethod` adds −0.300, and the other 15 features add +0.877 together. The sum −1.718 + 0.719 − 0.342 + 0.317 − 0.300 + 0.877 = −0.447 is exactly the model's log-odds, a churn probability of 1/(1 + e^0.447) = 39 %.

The mean absolute SHAP value per feature is a **global** importance; the individual values are **local** explanations.

### Why it matters: SHAP for checking and debugging

Use SHAP first as a tool for the modeller, not as a sales argument:

- **Find leaks.** A feature that should be weak but dominates the SHAP summary is the most common sign of target leakage (Session 9).
- **Find bugs.** Unexpected directions (a higher price *lowers* churn risk) or effects of an ID column point to errors in the feature pipeline.
- **Check single cases.** For a surprising prediction, the SHAP values show which inputs drove it, often revealing a data-quality problem in that row.
- **Find proxies.** A large contribution of a postcode or a name-derived feature may signal discrimination; that discussion belongs to the governance module, but SHAP is how you notice it.

### How it works in Python

In [9]:
import shap

explainer = shap.TreeExplainer(lgbm)
sv = explainer(Xc_te)                                   # Explanation object: values, base_values, data
print(sv.values.shape, round(float(sv.base_values[0]), 3))   # (1761, 19) -1.718
global_imp = pd.Series(np.abs(sv.values).mean(axis=0), index=X.columns).sort_values(ascending=False)
print(global_imp.head(5).round(3).to_dict())
# {'Contract': 0.821, 'tenure': 0.487, 'OnlineSecurity': 0.257, 'MonthlyCharges': 0.25, 'InternetService': 0.219}

i = 0                                                   # one customer: check additivity
p = lgbm.predict_proba(Xc_te.iloc[[i]])[0, 1]
print(round(float(sv.base_values[i] + sv.values[i].sum()), 3), round(float(np.log(p / (1 - p))), 3))
# -0.447 -0.447: base value + contributions = model log-odds
print(pd.Series(sv.values[i], index=X.columns).sort_values(key=abs, ascending=False).head(4).round(3).to_dict())
# {'Contract': 0.719, 'InternetService': -0.342, 'tenure': 0.317, 'PaymentMethod': -0.3}
# shap.plots.beeswarm(sv)   # one dot per customer and feature; shap.plots.waterfall(sv[i]) for one customer

.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(1761, 19) -1.718
{'Contract': 0.821, 'tenure': 0.487, 'OnlineSecurity': 0.257, 'MonthlyCharges': 0.25, 'InternetService': 0.219}
-0.447 -0.447
{'Contract': 0.719, 'InternetService': -0.342, 'tenure': 0.317, 'PaymentMethod': -0.3}


Now the debugging use. We add a feature `retention_offer` that marks customers who received a retention call. In this simulation, the call is made mostly *after* customers announced that they would leave, so the feature is recorded after the outcome: a leak.

In [10]:
leak = lambda yy: np.where(yy == 1, rng.random(len(yy)) < 0.6, rng.random(len(yy)) < 0.05).astype(int)
leak_tr, leak_te = Xc_tr.assign(retention_offer=leak(y_tr)), Xc_te.assign(retention_offer=leak(y_te))
leaky = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=8, verbose=-1, random_state=0)
leaky.fit(leak_tr, y_tr)
print(round(roc_auc_score(y_te, leaky.predict_proba(leak_te)[:, 1]), 3))       # 0.918: suspiciously good
sv_leak = shap.TreeExplainer(leaky)(leak_te)
print(pd.Series(np.abs(sv_leak.values).mean(axis=0), index=leak_te.columns)
      .sort_values(ascending=False).head(3).round(3).to_dict())
# {'retention_offer': 1.112, 'Contract': 0.698, 'tenure': 0.344}

0.918
{'retention_offer': 1.112, 'Contract': 0.698, 'tenure': 0.344}


The test AUC jumps from 0.842 to 0.918, and the SHAP summary shows one feature far ahead of contract type, which domain experts know as the strongest churn driver. The right reaction is not to celebrate but to ask *when* `retention_offer` is recorded. The same check on the Berlin price model of Section 9 would put Inside Airbnb's `estimated_revenue_l365d` (revenue = occupancy × price, Session 9) far ahead of every honest feature.

### In practice

- Lundberg et al. (2018) used SHAP values with gradient boosting to explain predictions of hypoxaemia risk during surgery to anaesthesiologists at the University of Washington, in *Nature Biomedical Engineering*.
- The SHAP example notebook [extras/16-shap-causal-caution.ipynb](extras/16-shap-causal-caution.ipynb), written by the SHAP authors, shows with a simulated subscription business why SHAP values must not be read as causal effects of changing a feature.

> **Caution.** SHAP values explain the model, not the customer's behaviour. "Month-to-month contract adds +0.72 log-odds" means the model's prediction rises; it does not mean that moving the customer to a yearly contract would lower *their* churn risk by that amount. Causal questions need experiments or causal methods.

> **Warning.** The `shap` package changes its API often. Pin the version, check shapes (multi-class models return one set of values per class, an extra dimension), and note whether values are on the log-odds or probability scale.

## 8. Comparing the performance of models

### Concept

In a running system there is always a **current model** (the *champion*), for example the logistic regression of Session 8. A new **challenger**, here the boosted model, should replace it only if it is better by a margin that is (a) larger than the noise of the evaluation and (b) worth the extra cost. A fair comparison requires:

1. **Same data, same splits.** Both models are evaluated on identical cross-validation folds and the identical test set, so that the comparison is **paired**: for each fold we look at the *difference* of the two scores.
2. **Uncertainty of the difference.** Report the mean difference with its fold-to-fold standard deviation, or a bootstrap confidence interval of the difference on the test set (Session 7). If the interval contains 0, the data do not show that the challenger is better.
3. **Costs beyond the metric.** Training and prediction time, memory, dependencies (OpenMP, GPU), explainability requirements, monitoring effort and the risk of a new pipeline.

Worked example: over 15 folds the challenger wins 12 times, with a mean difference in ROC AUC of +0.003 and a standard deviation of 0.004. The standard error of the mean is 0.004/√15 ≈ 0.001, so the improvement is probably real, but tiny: 0.003 AUC rarely changes a business decision.

### Why it matters

Without a disciplined comparison, teams replace working models with more complex ones on the strength of one lucky split. In production, model changes are usually staged: offline comparison as here, then a **shadow deployment** (the challenger predicts alongside the champion without affecting decisions) or an A/B test, and only then the switch (Session 16).

### How it works in Python

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num = X.select_dtypes("number").columns.tolist()
current = make_pipeline(ColumnTransformer([("num", StandardScaler(), num),
                                           ("cat", OneHotEncoder(handle_unknown="ignore"), cat)]),
                        LogisticRegression(max_iter=1000))
challenger = LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=8, min_child_samples=40,
                            n_jobs=1, verbose=-1, random_state=0)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)       # identical folds for both
s_cur = cross_val_score(current, Xc_tr, y_tr, cv=cv, scoring="roc_auc")
s_ch = cross_val_score(challenger, Xc_tr, y_tr, cv=cv, scoring="roc_auc")
d = s_ch - s_cur                                                            # paired differences
print(round(s_cur.mean(), 4), round(s_ch.mean(), 4), round(d.mean(), 4), round(d.std(ddof=1), 4),
      (d > 0).sum(), "of", len(d))
# 0.8452 0.848 0.0028 0.0037 12 of 15

# bootstrap confidence interval of the difference on the held-out test set
p_cur = current.fit(Xc_tr, y_tr).predict_proba(Xc_te)[:, 1]
p_ch = challenger.fit(Xc_tr, y_tr).predict_proba(Xc_te)[:, 1]
yt, rng_b, diffs = y_te.to_numpy(), np.random.default_rng(0), []
for _ in range(1000):
    idx = rng_b.integers(0, len(yt), len(yt))                               # resample test customers
    diffs.append(roc_auc_score(yt[idx], p_ch[idx]) - roc_auc_score(yt[idx], p_cur[idx]))
print(round(roc_auc_score(yt, p_cur), 3), round(roc_auc_score(yt, p_ch), 3),
      np.percentile(diffs, [2.5, 97.5]).round(3))
# 0.844 0.845 [-0.005  0.007]

0.8452 0.848 0.0028 0.0037 12 of 15


0.844 0.845 [-0.005  0.007]


The challenger wins 12 of 15 folds by 0.003 AUC on average. On the test set, the 95 % bootstrap interval of the difference, from −0.005 to +0.007, contains 0. The honest conclusion for the churn data: **keep the logistic regression**. It is as accurate, faster, has fewer dependencies, and its coefficients are easy to explain. On the Berlin price data the same procedure gives the opposite answer (next section): there the challenger wins in every fold, by a margin that matters to a host.

> **Tip.** Write the comparison down as a small table: metric with interval, training time, prediction time per 1,000 rows, model size, and one sentence on explainability. Decide with the table, not with the first line.

### In practice

- Bernardi et al. (2019) report from 150 models deployed at Booking.com that gains in offline metrics often did not translate into business gains, which is why model changes there are validated in randomised online experiments.
- In banking, model validation units compare challengers with the champion on the same data before a model change is approved, as expected by supervisory guidance on model risk (for example SR 11-7 in the United States).

> **Warning.** Comparing many challengers on the same test set and picking the best overfits the test set, just as tuning on it would. Use cross-validation to choose; use the test set once, for the final comparison.

## 9. The case study: what drives Berlin Airbnb prices, and do trees price better?

**Question.** A host in Berlin wants a price suggestion for a flat. The current model is the linear regression of Session 6: log price explained by guests, room type, distance to Alexanderplatz and district. The challenger is LightGBM on everything the listing page offers: size, bedrooms, bathrooms, location, property type, minimum stay, availability, reviews and 122 common amenities. The data are the 6,675 listings of Sessions 7 and 9: the short-stay table of Sessions 4–6 (minimum stay below 28 nights) without 26 outliers priced below €10 or above €1,000 (Inside Airbnb, snapshot of 26 June 2026). The Session 6 model is re-evaluated here on these folds, so its error differs from the test error reported in Session 6. Validation keeps each host's listings in one fold (Session 7), because many hosts offer several similar flats. Leaky columns such as `estimated_revenue_l365d` stay out (Session 9).

In [12]:
import json
import re

from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit

lst = pd.read_parquet("case-study/data/airbnb/listings.parquet")
bnb = lst[(lst["minimum_nights"] < 28) & lst["price"].between(10, 1000)].reset_index(drop=True)
bnb["dist_km"] = np.hypot((bnb["latitude"] - 52.5219) * 111.2, (bnb["longitude"] - 13.4132) * 68.0)
amen_lists = bnb["amenities"].map(json.loads)
common = amen_lists.explode().value_counts().loc[lambda c: c >= 200].index            # 122 common amenities
amen = pd.DataFrame({"am_" + re.sub(r"\W+", "_", a).strip("_").lower(): amen_lists.map(lambda x, a=a: a in x).astype(int)
                     for a in common})                                                # LightGBM needs plain names
y, price, hosts = np.log(bnb["price"]), bnb["price"], bnb["host_id"]

X_s6 = pd.get_dummies(bnb[["accommodates", "dist_km", "room_type", "district"]], dtype=int)  # Session 6 features
rich_num = ["accommodates", "bedrooms", "beds", "bathrooms", "dist_km", "latitude", "longitude",
            "minimum_nights", "availability_365", "number_of_reviews", "review_scores_rating"]
X_rich = pd.concat([bnb[rich_num], bnb[["room_type", "district", "property_type"]].astype("category"),
                    amen.assign(n_amenities=amen_lists.map(len))], axis=1)
X_dummies = pd.get_dummies(X_rich, dtype=int)                                         # for ridge and the forest


def lgbm():
    return LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=31, subsample=0.8, subsample_freq=1,
                         colsample_bytree=0.5, verbose=-1, random_state=0)


models = {
    "current: linear, Session 6 features": (LinearRegression(), X_s6),
    "LightGBM, Session 6 features": (lgbm(), X_s6),
    "ridge, all features": (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                                          Ridge(alpha=10)), X_dummies),
    "random forest, all features": (RandomForestRegressor(300, min_samples_leaf=3, max_features=0.33,
                                                          n_jobs=-1, random_state=0), X_dummies),
    "challenger: LightGBM, all features": (lgbm(), X_rich),
}
fold_mae, rows = {}, []
for name, (model, X_m) in models.items():
    pred, fold_mae[name] = np.zeros(len(y)), []
    for tr, va in GroupKFold(5).split(X_m, y, groups=hosts):                          # identical folds for all
        pred[va] = model.fit(X_m.iloc[tr], y.iloc[tr]).predict(X_m.iloc[va])
        fold_mae[name].append(mean_absolute_error(price.iloc[va], np.exp(pred[va])))
    rows.append({"model": name, "R² (log)": r2_score(y, pred), "MAE €": mean_absolute_error(price, np.exp(pred)),
                 "median AE €": np.median(np.abs(price - np.exp(pred)))})
print(pd.DataFrame(rows).set_index("model").round(3).to_string())
d = np.array(fold_mae["challenger: LightGBM, all features"]) - np.array(fold_mae["current: linear, Session 6 features"])
print(d.round(1), round(d.mean(), 1))         # paired differences of the MAE per fold, in euros

                                     R² (log)   MAE €  median AE €
model                                                             
current: linear, Session 6 features     0.506  56.935       35.761
LightGBM, Session 6 features            0.498  57.141       37.422
ridge, all features                     0.616  49.635       31.750
random forest, all features             0.614  48.379       30.472
challenger: LightGBM, all features      0.637  46.405       28.886
[ -9.6 -12.4  -8.8 -11.5 -10.3] -10.5


| Model | R² (log price) | MAE € | Median absolute error € |
|---|---|---|---|
| current: linear, Session 6 features | 0.506 | 56.9 | 35.8 |
| LightGBM, Session 6 features | 0.498 | 57.1 | 37.4 |
| ridge, all features | 0.616 | 49.6 | 31.8 |
| random forest, all features | 0.616 | 48.3 | 30.4 |
| challenger: LightGBM, all features | 0.635 | 46.5 | 29.0 |

The paired MAE differences of the challenger against the current model are −10.0, −12.3, −8.4, −11.2 and −10.3 euros: the challenger wins in every fold, by €10.4 a night on average. Here the comparison of Section 8 says **replace**. The table also says *why*. With the four Session 6 features, LightGBM is no better than the linear model (MAE €57.1 against €56.9): there is little structure for a flexible model to find. Most of the gain comes from the **features**: ridge on all features already reaches €49.6. The tree ensembles add the rest (€48.3 for the forest, €46.5 for LightGBM) by finding interactions and thresholds that a linear model would have to be given by hand. A typical listing is still mispriced by about €29 (median absolute error), which is honest information for a host: the model gives a starting point, not a price.

Which features drive the challenger, and is any of them suspicious? Permutation importance on held-out hosts and SHAP values on the log scale:

In [13]:
tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=0).split(X_rich, groups=hosts))   # held-out hosts
challenger = lgbm().fit(X_rich.iloc[tr], y.iloc[tr])
perm = permutation_importance(challenger, X_rich.iloc[te], y.iloc[te], scoring="r2", n_repeats=5, random_state=0)
imp = pd.Series(perm.importances_mean, index=X_rich.columns)
print(imp.sort_values(ascending=False).head(6).round(3).to_dict())
print("all 122 amenities together:", round(imp.filter(like="am_").sum(), 3))

sv = shap.TreeExplainer(challenger)(X_rich.iloc[te])               # SHAP values on the log-price scale
print(pd.Series(np.abs(sv.values).mean(axis=0), index=X_rich.columns).nlargest(5).round(3).to_dict())
i = 0                                                               # one held-out listing
print(X_rich.iloc[te[i]][["room_type", "accommodates", "bedrooms", "dist_km"]].to_dict())
print(round(float(np.exp(sv.base_values[i]))), round(float(np.exp(sv.base_values[i] + sv.values[i].sum()))),
      round(float(price.iloc[te[i]])))                             # base price, prediction, actual price in EUR
top = pd.Series(sv.values[i], index=X_rich.columns).sort_values(key=abs, ascending=False).head(4)
print((np.exp(top) - 1).round(3).to_dict())                        # each contribution as a price factor

{'accommodates': 0.139, 'property_type': 0.104, 'bedrooms': 0.09, 'dist_km': 0.032, 'minimum_nights': 0.026, 'number_of_reviews': 0.022}
all 122 amenities together: 0.04


{'accommodates': 0.122, 'property_type': 0.119, 'bedrooms': 0.115, 'dist_km': 0.055, 'room_type': 0.046}
{'room_type': 'Entire home/apt', 'accommodates': 2, 'bedrooms': 1.0, 'dist_km': 1.5645443855839263}
158 188 161
{'accommodates': -0.098, 'dist_km': 0.103, 'property_type': 0.102, 'availability_365': 0.083}


Shuffling `accommodates` costs 0.138 of R², property type 0.103, bedrooms 0.092, the distance to the centre 0.030; all 122 amenities together are worth 0.042. Nothing dominates suspiciously, and the order matches what a host would expect: size first, then the kind of property and location. The SHAP values agree. Because the model predicts the log price, a SHAP value φ is a factor on the price: e^φ. For the first held-out listing, an entire flat for two guests with one bedroom 1.6 km from Alexanderplatz, the base value is €158 (the average prediction) and the model predicts €188 (actual price: €161). Its small size lowers the prediction by 10 %; its property type raises it by 11 %, the central location by 10 % and its availability by 9 %. The factors multiply: SHAP values add on the log scale. Across all held-out listings (figure, right), the location contribution falls steadily with distance: a premium of up to about 25 % near Alexanderplatz, a discount of about 9 % beyond 8 km.

![Interpretation of the LightGBM price model on held-out hosts. Left: permutation importance (drop in R²) of the ten most important features; guests, property type and bedrooms lead. Right: SHAP value of the distance to Alexanderplatz, as a price factor, against the distance; listings within about 4 km of the centre get a premium of up to about 25 %, listings beyond 8 km a discount of about 9 %, with a spread of a few percentage points at a given distance (interactions with other features).](../figures/price_model_interpretation.png)

> **Caution.** "Central location raises the price by 10 %" describes the **model**. It is not the causal effect of moving a flat, and it is not advice to hosts to charge more. Location also stands for flat quality, building age and the kind of guest, which the data do not record.

## Check your understanding

1. What is the main difference between level-wise (XGBoost) and leaf-wise (LightGBM) tree growth, and which parameter controls the size of a LightGBM tree?
2. How does CatBoost encode a category without leaking the row's own label?
3. With 900 stayers and 300 churners, what is `scale_pos_weight`? Why does ROC AUC barely change when you set it?
4. Why must the early-stopping validation set be split off the training data, not the test data?
5. A search reports `best_score_` = 0.850 and the test score is 0.847. Why is the first number usually the higher one?
6. Why can a column of random numbers get a high impurity-based importance in a random forest, but not a high permutation importance on test data?
7. Two features are strongly correlated. What happens to their permutation importances, and how can you deal with it?
8. A SHAP base value is −1.0 and the contributions of a customer are +0.8, −0.3 and +0.5. What is the model's log-odds and probability for this customer?
9. A new feature makes the test AUC jump from 0.84 to 0.92 and dominates the SHAP summary. What do you check before using it?
10. A challenger wins 9 of 15 folds with a mean AUC difference of +0.001. Would you replace the current model? What else would you consider?
11. With the four Session 6 features, LightGBM is no better than the linear model; with all features it is clearly better. What does that tell you about where the gain of a tree ensemble comes from?

## Further reading

- Chen, T. & Guestrin, C. (2016). XGBoost: a scalable tree boosting system. *Proceedings of KDD 2016*, 785–794. https://doi.org/10.1145/2939672.2939785
- Ke, G. et al. (2017). LightGBM: a highly efficient gradient boosting decision tree. *Advances in Neural Information Processing Systems 30*. https://papers.nips.cc/paper_files/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html
- Prokhorenkova, L., Gusev, G., Vorobev, A., Dorogush, A. V. & Gulin, A. (2018). CatBoost: unbiased boosting with categorical features. *Advances in Neural Information Processing Systems 31*. https://arxiv.org/abs/1706.09516
- LightGBM developers (2025). *Parameters Tuning*. LightGBM documentation. https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html
- Molnar, C. (2025). *Interpretable Machine Learning*, 3rd edition, chapters on permutation feature importance and SHAP. https://christophm.github.io/interpretable-ml-book/
- Lundberg, S. M. & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems 30*. https://arxiv.org/abs/1705.07874
- scikit-learn developers (2025). *Permutation Importance vs Random Forest Feature Importance (MDI)*. scikit-learn example gallery. https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance.html
- Bernardi, L., Mavridis, T. & Estevez, P. (2019). 150 successful machine learning models: 6 lessons learned at Booking.com. *Proceedings of KDD 2019*, 1743–1751. https://doi.org/10.1145/3292500.3330744

> **Note.** For text the picture reverses: on the customs decisions of Sessions 13–16, a linear classifier on sparse TF-IDF features (Session 13) is far stronger than boosting on compressed text.